In [9]:
import anthropic
import json
import csv
import time
from pathlib import Path
import pandas as pd

client = anthropic.Anthropic()


In [10]:
JUDGE_PROMPT = """You are an expert judge evaluating knowledge graph reasoning quality.

You will be given:
1. A summary triple — a claim extracted from a document summary
2. A document knowledge graph — a set of triplets extracted from the source document
3. A predicted supporting evidence — the reasoning chain the model used to support or reject the claim

Your job is to evaluate the quality of the predicted supporting evidence on four dimensions.

---

SUMMARY TRIPLE:
{summary_triple}

DOCUMENT KNOWLEDGE GRAPH:
{doc_triplets}

PREDICTED SUPPORTING EVIDENCE:
{supporting_evidence}

PREDICTED JUDGMENT: {is_supported}
PREDICTED REASONING: {reasoning}

---

Evaluate the supporting evidence on the following four dimensions.
For each dimension, give a score and a one-sentence justification.

1. ENTITY RESOLUTION (0-2)
   - 0: Model failed to match summary entities to KG entities
   - 1: Partial match — found some but missed key entity equivalences
   - 2: Correctly resolved all relevant entity mentions to KG nodes

2. PATH VALIDITY (0-2)
   - 0: Path contains edges that don't exist in the document KG
   - 1: Path exists in the KG but skips intermediate nodes or uses inverse edges incorrectly
   - 2: Every step in the path is a valid edge in the document KG

3. PATH RELEVANCE (0-2)
   - 0: Path is unrelated to the summary triple's claim
   - 1: Path is topically related but doesn't directly bridge the summary claim
   - 2: Path directly and sufficiently connects the summary triple's head to its tail

4. JUDGMENT CONSISTENCY (0-2)
   - 0: The is_supported judgment directly contradicts what the supporting evidence shows
   - 1: Judgment is plausible but the reasoning doesn't fully justify it
   - 2: Judgment is correct and the reasoning clearly follows from the supporting evidence

---

Respond ONLY in the following JSON format with no preamble:
{{
  "entity_resolution": {{"score": <0-2>, "justification": "<one sentence>"}},
  "path_validity": {{"score": <0-2>, "justification": "<one sentence>"}},
  "path_relevance": {{"score": <0-2>, "justification": "<one sentence>"}},
  "judgment_consistency": {{"score": <0-2>, "justification": "<one sentence>"}},
  "total_score": <0-8>,
  "overall_verdict": "<GOOD | ACCEPTABLE | POOR>",
  "failure_mode": "<null | ENTITY_MISMATCH | HALLUCINATED_EDGE | IRRELEVANT_PATH | WRONG_JUDGMENT | INCOMPLETE_CHAIN>"
}}
"""

In [11]:
def prepare_batch_requests(csv_file_path, output_jsonl_path, start_index=0, end_index=623):
    """
    Convert CSV data into batch API JSONL format.
    Each line is a separate request for KG triplet extraction.
    
    Args:
        csv_file_path: Path to input CSV file
        output_jsonl_path: Path to output JSONL file
        start_index: Starting row index (0-based, inclusive)
        end_index: Ending row index (0-based, exclusive)
    """
    with open(csv_file_path, 'r', encoding='utf-8') as csv_file, \
         open(output_jsonl_path, 'w', encoding='utf-8') as jsonl_file:
        
        reader = csv.DictReader(csv_file)
        
        for idx, row in enumerate(reader):
            # Skip rows before start_index
            if idx < start_index:
                continue
            # Stop after end_index
            if idx >= end_index:
                break
                
            text_id = idx  # Adjust column name as needed
            text_triplets = row['document_triplets']  # Adjust column name as needed
            summary_triplet = row['summary_triplet_full']  # Adjust column name as needed
            supporting_evidence = row['supporting_evidence']  # Adjust column name as needed
            is_supported = row['is_supported']  # Adjust column name as needed
            reasoning = row['reasoning']  # Adjust column name as needed
            
            # Create the batch request
            batch_request = {
                "custom_id": f"kg-extraction-{text_id}",
                "params": {
                    "model": "claude-sonnet-4-6",
                    # "model": "claude-sonnet-4-5-20250929",
                    "max_tokens": 20000,
                    'temperature': 0.1,
                    "system": "You are an expert evaluator specializing in knowledge graph reasoning and natural language inference",
                    "messages": [
                        {
                            "role": "user",
                            "content": JUDGE_PROMPT.format(
                                summary_triple=summary_triplet,
                                doc_triplets=text_triplets,
                                supporting_evidence=supporting_evidence,
                                is_supported=is_supported,
                                reasoning=reasoning
                            )
                        }
                    ]
                }
            }
            
            jsonl_file.write(json.dumps(batch_request) + '\n')
    
    print(f"Batch requests prepared and saved to {output_jsonl_path}")

In [4]:
def submit_batch(jsonl_file_path):
    """
    Submit the batch of requests to Claude API.
    Returns the batch ID for tracking.
    """
    # Read all requests from the JSONL file
    requests = []
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            requests.append(json.loads(line))
    
    # Create the message batch
    message_batch = client.messages.batches.create(
        requests=requests
    )
    
    print(f"Batch submitted successfully!")
    print(f"Batch ID: {message_batch.id}")
    print(f"Status: {message_batch.processing_status}")
    
    return message_batch.id

def check_batch_status(batch_id):
    """
    Check the status of a batch processing job.
    """
    batch = client.messages.batches.retrieve(batch_id)
    
    print(f"\nBatch Status: {batch.processing_status}")
    print(f"Requests: {batch.request_counts}")
    
    return batch.processing_status

def retrieve_batch_results(batch_id, output_jsonl_path):
    """
    Retrieve results once batch processing is complete.
    """
    # Get all results
    results = client.messages.batches.results(batch_id)
    
    with open(output_jsonl_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result.model_dump()) + '\n')
    
    print(f"Results saved to {output_jsonl_path}")

def process_results(results_jsonl_path, output_csv_path):
    """
    Process the batch results and extract KG triplets into a CSV.
    """
    triplets = []
    
    with open(results_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            result = json.loads(line)
            
            # Extract custom_id to get original text ID
            custom_id = result['custom_id']
            text_id = custom_id.replace('kg-extraction-', '')
            
            # Check if request was successful
            if result['result']['type'] == 'succeeded':
                response = result['result']['message']
                
                # Extract the triplets from the response
                content = response['content'][0]['text']
                
                try:
                    # Parse the JSON response
                    extracted_triplets = json.loads(content)
                    
                    # Add text_id to each triplet
                    for triplet in extracted_triplets:
                        triplets.append({
                            'text_id': text_id,
                            'subject': triplet['subject'],
                            'predicate': triplet['predicate'],
                            'object': triplet['object']
                        })
                except json.JSONDecodeError:
                    print(f"Warning: Could not parse response for {custom_id}")
            else:
                print(f"Error processing {custom_id}: {result['result']}")
    
    # Write to CSV
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['text_id', 'subject', 'predicate', 'object']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(triplets)
    
    print(f"Processed {len(triplets)} triplets and saved to {output_csv_path}")


In [12]:
def main():
    """
    Main workflow for batch KG extraction.
    """
    #     # File paths
    input_csv = "../data/test/qwen3/04-03-26/results_traversal_ck_300_t_01_v2_expanded.csv"
    batch_requests_file = "./qwen3_8b_lora/batch_requests_traversal_1.jsonl"
    batch_results_file = "./qwen3_8b_lora/batch_results_traversal_1.jsonl"

    # Step 1: Prepare batch requests
    print("Step 1: Preparing batch requests...")
    prepare_batch_requests(input_csv, batch_requests_file)
    
    # Step 2: Submit batch
    print("\nStep 2: Submitting batch...")
    batch_id = submit_batch(batch_requests_file)
    
    # Step 3: Poll for completion
    print("\nStep 3: Waiting for batch to complete...")
    while True:
        status = check_batch_status(batch_id)
        
        if status == "ended":
            break
        
        print("Still processing... checking again in 60 seconds")
        time.sleep(60)
    
    # Step 4: Retrieve results
    print("\nStep 4: Retrieving results...")
    retrieve_batch_results(batch_id, batch_results_file)



In [13]:
if __name__ == "__main__":
    main()

Step 1: Preparing batch requests...
Batch requests prepared and saved to ./qwen3_8b_lora/batch_requests_traversal_1.jsonl

Step 2: Submitting batch...
Batch submitted successfully!
Batch ID: msgbatch_01HbPjL66hsKd6trg7XTESc1
Status: in_progress

Step 3: Waiting for batch to complete...

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=623, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=623, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=623, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status: in_progress
Requests: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=623, succeeded=0)
Still processing... checking again in 60 seconds

Batch Status

KeyboardInterrupt: 

In [1]:
# merge the two jsonl file to create a final json file with all the results for analysis
from pathlib import Path
import json

def merge_jsonl_files(input_files, output_file):
    seen = set()
    merged = []

    for file_path in input_files:
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                custom_id = record.get("custom_id")
                if custom_id and custom_id in seen:
                    continue
                if custom_id:
                    seen.add(custom_id)
                merged.append(record)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(merged, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(merged)} records to {output_file}")

base_dir = Path("/Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/eval")
input_files = [
    base_dir / "qwen3_8b_lora/batch_results_traversal_1.jsonl",
    # base_dir / "qwen3_q/batch_results_traversal_2.jsonl",
]
output_file = base_dir / "qwen3_8b_lora/batch_results_final.json"

merge_jsonl_files(input_files, output_file)

Saved 623 records to /Users/nirashanelki/Documents/Works/Research/codes/factual_inconsistency_validate_xai/eval/qwen3_8b_lora/batch_results_final.json


## eval and analaysis

In [2]:
"""
LLM-as-Judge Evaluation Script
================================
Collects batch results, computes all metrics, saves to JSON.

Usage:
    # from a completed batch
    # set results_source = "batch_id" and batch_id = "msgbatch_xxxxx"
    #
    # from already-saved batch results
    # set results_source = "results_file" and results_file = "batch_results.json"
    #
    # with gold labels for classification metrics
    # set gold_labels_file = "gold.jsonl"
"""

import json
import numpy as np
from collections import Counter, defaultdict
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, confusion_matrix,
    balanced_accuracy_score
)

DIMS = ["entity_resolution", "path_validity", "path_relevance", "judgment_consistency"]
FAILURE_MODES = ["null", "ENTITY_MISMATCH", "HALLUCINATED_EDGE",
                 "IRRELEVANT_PATH", "WRONG_JUDGMENT", "INCOMPLETE_CHAIN"]


# ---------------------------------------------------------------------------
# Loaders
# ---------------------------------------------------------------------------

def _parse_result_record(record):
    custom_id = record.get("custom_id") if isinstance(record, dict) else None
    if not custom_id:
        return None, None
    result = record.get("result") if isinstance(record, dict) else None
    if not result or result.get("type") != "succeeded":
        return custom_id, None
    message = result.get("message") or {}
    content = message.get("content") or []
    if not content or not isinstance(content, list):
        return custom_id, None
    raw = content[0].get("text", "")
    try:
        clean = raw.strip().removeprefix("```json").removesuffix("```").strip()
        return custom_id, json.loads(clean)
    except json.JSONDecodeError:
        return custom_id, None


def load_from_batch(batch_id):
    import anthropic
    client = anthropic.Anthropic()
    results = {}
    for result in client.messages.batches.results(batch_id):
        cid = result.custom_id
        if result.result.type == "succeeded":
            raw = result.result.message.content[0].text
            try:
                clean = raw.strip().removeprefix("```json").removesuffix("```").strip()
                results[cid] = json.loads(clean)
            except json.JSONDecodeError:
                results[cid] = None
        else:
            results[cid] = None
    return results


def load_from_file(path):
    with open(path) as f:
        data = json.load(f)

    if isinstance(data, dict):
        return data
    if not isinstance(data, list):
        raise ValueError("results_file must be a dict or list JSON")

    results = {}
    for record in data:
        cid, parsed = _parse_result_record(record)
        if cid is None:
            continue
        results[cid] = parsed
    return results


def load_gold(path):
    """
    JSONL where each line has at minimum:
      {"custom_id": "eval_0", "is_supported": true, "confidence": 2}
    """
    gold = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line.strip())
            gold[r["custom_id"]] = r
    return gold


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------

def dimension_stats(scored):
    out = {}
    for dim in DIMS:
        vals = [r[dim]["score"] for r in scored if dim in r]
        out[dim] = {
            "mean":         round(float(np.mean(vals)), 4),
            "std":          round(float(np.std(vals)), 4),
            "median":       round(float(np.median(vals)), 4),
            "pct_perfect":  round(sum(v == 2 for v in vals) / len(vals), 4),
            "pct_zero":     round(sum(v == 0 for v in vals) / len(vals), 4),
            "distribution": dict(Counter(vals)),
        }
    return out


def total_score_stats(scored):
    vals = [r["total_score"] for r in scored]
    return {
        "mean":           round(float(np.mean(vals)), 4),
        "std":            round(float(np.std(vals)), 4),
        "median":         round(float(np.median(vals)), 4),
        "min":            int(min(vals)),
        "max":            int(max(vals)),
        "pct_good":       round(sum(v >= 6 for v in vals) / len(vals), 4),
        "pct_acceptable": round(sum(4 <= v <= 5 for v in vals) / len(vals), 4),
        "pct_poor":       round(sum(v <= 3 for v in vals) / len(vals), 4),
        "distribution":   {str(k): v for k, v in sorted(Counter(vals).items())},
    }


def verdict_stats(scored):
    counts = Counter(r["overall_verdict"] for r in scored)
    n = len(scored)
    return {v: {"count": counts.get(v, 0),
                "pct":   round(counts.get(v, 0) / n, 4)}
            for v in ["GOOD", "ACCEPTABLE", "POOR"]}


def failure_mode_stats(scored):
    all_modes    = Counter(r.get("failure_mode", "null") for r in scored)
    failed       = [r for r in scored if r["total_score"] < 6]
    failed_modes = Counter(r.get("failure_mode", "null") for r in failed)
    n, nf = len(scored), max(len(failed), 1)
    return {
        "top_failure_mode": failed_modes.most_common(1)[0][0] if failed_modes else "null",
        "all_cases": {m: {"count": all_modes.get(m, 0),
                          "pct":   round(all_modes.get(m, 0) / n, 4)}
                      for m in FAILURE_MODES},
        "failed_cases_only": {m: {"count": failed_modes.get(m, 0),
                                   "pct":   round(failed_modes.get(m, 0) / nf, 4)}
                               for m in FAILURE_MODES},
    }


def correlation_matrix(scored):
    data = {d: [r[d]["score"] for r in scored] for d in DIMS}
    data["total"] = [r["total_score"] for r in scored]
    keys = DIMS + ["total"]
    return {
        k1: {k2: round(float(np.corrcoef(data[k1], data[k2])[0, 1]), 4)
             for k2 in keys}
        for k1 in keys
    }


def stratified_stats(scored):
    strata = defaultdict(list)
    for r in scored:
        label = r.get("_is_supported")
        conf  = r.get("_confidence")
        score = r["total_score"]
        if label is not None:
            strata[f"is_supported_{label}"].append(score)
        if conf is not None:
            strata[f"confidence_{conf}"].append(score)
        if label is not None and conf is not None:
            strata[f"supported_{label}_conf_{conf}"].append(score)
    return {
        k: {"n":    len(v),
            "mean": round(float(np.mean(v)), 4),
            "std":  round(float(np.std(v)), 4)}
        for k, v in strata.items()
    }


def classification_metrics(scored, gold):
    pred, true, scores = [], [], []
    for r in scored:
        cid = r.get("_custom_id")
        if not cid or cid not in gold:
            continue
        pred.append(1 if r["total_score"] >= 4 else 0)
        true.append(1 if gold[cid]["is_supported"] else 0)
        scores.append(r["total_score"])

    if len(pred) < 2:
        return {}

    cm = confusion_matrix(true, pred).tolist()
    correct   = [s for s, p, g in zip(scores, pred, true) if p == g]
    incorrect = [s for s, p, g in zip(scores, pred, true) if p != g]

    return {
        "n_matched":                len(pred),
        "accuracy":                 round(accuracy_score(true, pred), 4),
        "balanced_accuracy":        round(balanced_accuracy_score(true, pred), 4),
        "precision":                round(precision_score(true, pred, zero_division=0), 4),
        "recall":                   round(recall_score(true, pred, zero_division=0), 4),
        "f1":                       round(f1_score(true, pred, zero_division=0), 4),
        "specificity":              round(cm[0][0] / max(cm[0][0] + cm[0][1], 1), 4),
        "cohen_kappa":              round(cohen_kappa_score(true, pred), 4),
        "confusion_matrix":         cm,
        "avg_score_correct_pred":   round(float(np.mean(correct)), 4)   if correct   else None,
        "avg_score_incorrect_pred": round(float(np.mean(incorrect)), 4) if incorrect else None,
    }


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def build_report(results_dict, gold=None):
    scored = []
    for cid, r in results_dict.items():
        if r is None:
            continue
        r["_custom_id"] = cid
        if gold and cid in gold:
            r["_is_supported"] = gold[cid].get("is_supported")
            r["_confidence"]   = gold[cid].get("confidence")
        scored.append(r)

    n_total  = len(results_dict)
    n_scored = len(scored)

    print(f"  Total submitted:     {n_total}")
    print(f"  Successfully parsed: {n_scored}")
    print(f"  Failed/unparseable:  {n_total - n_scored}")

    report = {
        "meta": {
            "n_total":    n_total,
            "n_scored":   n_scored,
            "n_failed":   n_total - n_scored,
            "parse_rate": round(n_scored / max(n_total, 1), 4),
        },
        "total_score":   total_score_stats(scored),
        "dimensions":    dimension_stats(scored),
        "verdicts":      verdict_stats(scored),
        "failure_modes": failure_mode_stats(scored),
        "correlations":  correlation_matrix(scored),
        "stratified":    stratified_stats(scored),
    }

    if gold:
        report["classification"] = classification_metrics(scored, gold)

    return report


def print_summary(report):
    ts = report["total_score"]
    print(f"\n  Mean total score : {ts['mean']:.3f} / 8  (σ={ts['std']:.3f})")
    print(f"  Good  (≥6)       : {ts['pct_good']*100:.1f}%")
    print(f"  Acceptable (4-5) : {ts['pct_acceptable']*100:.1f}%")
    print(f"  Poor  (≤3)       : {ts['pct_poor']*100:.1f}%")
    print(f"\n  Dimension means:")
    for dim, s in report["dimensions"].items():
        print(f"    {dim:<28} {s['mean']:.3f}  (perfect={s['pct_perfect']*100:.0f}%)")
    print(f"\n  Top failure mode : {report['failure_modes']['top_failure_mode']}")
    if "classification" in report and report["classification"]:
        c = report["classification"]
        print(f"\n  vs gold labels (n={c['n_matched']})")
        print(f"    Accuracy  {c['accuracy']*100:.1f}%  |  F1 {c['f1']*100:.1f}%  |  "
              f"Recall {c['recall']*100:.1f}%  |  κ={c['cohen_kappa']:.3f}")


# -----------------------------
# Configure inputs here
# -----------------------------

results_source = "results_file"  # "batch_id" or "results_file"
batch_id = None  # e.g. "msgbatch_xxxxx"
results_file = "./qwen3_8b_lora/batch_results_final.json"  # e.g. "batch_results.json"
gold_labels_file = None  # e.g. "gold.jsonl"
output_path = "./qwen3_8b_lora/eval_results.json"


print("Loading results...")
if results_source == "batch_id":
    if not batch_id:
        raise ValueError("batch_id is required when results_source='batch_id'.")
    results = load_from_batch(batch_id)
elif results_source == "results_file":
    if not results_file:
        raise ValueError("results_file is required when results_source='results_file'.")
    results = load_from_file(results_file)
else:
    raise ValueError("results_source must be 'batch_id' or 'results_file'.")

gold = load_gold(gold_labels_file) if gold_labels_file else None
if gold:
    print(f"Loaded {len(gold)} gold labels")

print("\nComputing metrics...")
report = build_report(results, gold)
print_summary(report)

with open(output_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nSaved to {output_path}")

Loading results...

Computing metrics...
  Total submitted:     623
  Successfully parsed: 622
  Failed/unparseable:  1

  Mean total score : 5.547 / 8  (σ=1.531)
  Good  (≥6)       : 43.9%
  Acceptable (4-5) : 48.1%
  Poor  (≤3)       : 8.0%

  Dimension means:
    entity_resolution            1.233  (perfect=29%)
    path_validity                1.841  (perfect=87%)
    path_relevance               1.040  (perfect=25%)
    judgment_consistency         1.433  (perfect=47%)

  Top failure mode : INCOMPLETE_CHAIN

Saved to ./qwen3_8b_lora/eval_results.json
